In [41]:
import pandas as pd
import re
from datetime import datetime, date, timedelta
from calendar import monthrange

In [2]:
df= pd.DataFrame()
df= pd.read_json("../data/processed/processed_events_2026-01-31.json")
df.shape

(8216, 14)

In [34]:
def normalize_str(text:str) -> str:
    location = text.strip().lower()
    """ Remove accents from text """
    accents = { 'a': ['à', 'ã', 'á', 'â'],
                'e': ['é', 'è', 'ê', 'ë'],
                'i': ['î', 'ï'],
                'u': ['ù', 'ü', 'û'],
                'o': ['ô', 'ö'],
                ' ': ['-','/'] 
                }
    for (char, accented_chars) in accents.items():
        for accented_char in accented_chars:
            location = location.replace(accented_char, char)
    return location    

In [ ]:

def parse_date(query: str):
    """Extract date from query"""
    months = ['janvier',"fevrier", "mars", "avril", "mai","juin", "juillet","aout", "septembre", "octobre", "novembre", "decembre"]
    query_lower = normalize_str(query)
    today = today = datetime.today()

    #On a precise date
    match = re.search(r'\ble\s+(0?[1-9]|[12][0-9]|3[01])[\/\-](0?[1-9]|1[0,1,2])[\/\-\s](?:20|)([234][0-9]|)', query_lower +" ")
    if match and match.lastindex >= 2: # pyright: ignore[reportOptionalOperand]
        dd = int(match[1])
        mm = int(match[2])
        aa = int("20" + match[3] if match[3] else datetime.now().strftime('%y'))
        return (datetime(aa,mm,dd),0)

    # Today/tonight
    if re.search(r'\b(ce soir|aujourd\'hui|ce jour|ce matin|cet[te]* apres[\- ]midi)\b', query_lower):
        return (today,0)
    
    # Tomorrow
    if re.search(r'\b(demain)\b', query_lower):
        return (today+ timedelta(days=1), 0)
    
    # Relative days: "dans X jours"
    match = re.search(r'\bdans\s+(\d+)\s+(jour|jours)\b', query_lower)
    if match:
        days = int(match.group(1))
        return (today + timedelta(days=days), 0)
    
    # This weekend
    if re.search(r'\b(ce week[\- ]?end)\b', query_lower):
        days_until_saturday = (5 - today.weekday()) % 7 or 7
        
        return (today + timedelta(days=days_until_saturday) ,1)
    
    # Next week
    if re.search(r'\b(semaine\sprochaine)\b', query_lower):
        days_until_monday = (- today.weekday()) % 7 or 7
            
        return (today + timedelta(days=days_until_monday) ,6)
    
    # Current week
    if re.search(r'\b(?:cette|la)\s(semaine)\b', query_lower):
        days_until_sunday = (6 - today.weekday()) % 7 
            
        return (today  , days_until_sunday)
    
    # On precise month
    for index, month in enumerate(months):
        current_year = int(today.strftime("%Y"))
        if re.search(fr'\b(?:mois de|en)\s({month})\b', query_lower):
            index_month = (index + 1) 
            first_day = datetime(current_year, index_month,1)
            days_in_months = monthrange(first_day.year, first_day.month)[1]
            
            return (first_day ,days_in_months-1)
            

    # dafault : return the next 30 days
    return (today, 30)


In [91]:

query_lower = "Un concert cette semaine "
print(parse_date(query_lower))


(datetime.datetime(2026, 2, 4, 15, 2, 27, 575276), 4)


In [4]:
dt = date(2026,2,2)
pattern = 'emploi'
selected_ids = []
for index,data in df.iterrows():
    has_pattern = pattern in data['title_fr'].lower() or pattern in data['description_fr'].lower() or pattern in data['longdescription_fr'].lower() 
    if has_pattern:
        for timing in data['timings']:
            date_timing = datetime.fromisoformat(timing['begin']).date()
            if date_timing == dt:
                selected_ids.append(data['uid'])
                break
if selected_ids :
    print(f"Réponse(s) trouvées : {len(selected_ids)}")
else:
    print("Pas de réponses")

Réponse(s) trouvées : 4


In [5]:
selected_events = df[df['uid'].isin(selected_ids)]
if not selected_events.empty:
    for idx, data in selected_events.iterrows():
        display(f"uid : {data['uid']}")
        display(f"title_fr : {data['title_fr']}")
        display(f"description_fr : {data['description_fr']}")
        display(f"longdescription_fr : {data['longdescription_fr']}")
        display(f"timings : {data['timings']}")                        

'uid : 45068906'

'title_fr : Rencontre Transport & Logistique, BTP, Sécurité - ECF BOUSCAREN'

"description_fr : Vous êtes en recherche d'emploi ou en projet de reconversion? Vous êtes intéressé par les métiers du Transport & de la Logistique, du BTP et vous souhaitez avoir de plus amples informations."

"longdescription_fr : <p>Vous êtes en recherche d'emploi ou en projet de reconversion? Vous êtes intéressé par les métiers du Transport &amp; de la Logistique, du BTP et vous souhaitez avoir de plus amples informations. Nous vous proposons de venir rencontrer des conseillers du centre de formation Bouscaren au sein de l'agence France Travail de Pérols.</p>\n<p>venez au moment où vous le souhaitez, entre 10h30 et 12h pour bénéficier d'un échange individuel avec des conseillers de Bouscaren. Merci de valider votre présence en vous i</p>"

"timings : [{'begin': '2026-02-02T10:30:00+01:00', 'end': '2026-02-02T12:00:00+01:00'}]"

...

'uid : 95892978'

'title_fr : Rencontre Transport & Logistique, BTP, Sécurité - ECF BOUSCAREN'

"description_fr : Vous êtes en recherche d'emploi ou en projet de reconversion? Vous êtes intéressé par les métiers du Transport & de la Logistique, du BTP et vous souhaitez avoir de plus amples informations."

"longdescription_fr : <p>Vous êtes en recherche d'emploi ou en projet de reconversion? Vous êtes intéressé par les métiers du Transport &amp; de la Logistique, du BTP et vous souhaitez avoir de plus amples informations. Nous vous proposons de venir rencontrer des conseillers du centre de formation Bouscaren au sein de l'agence France Travail de Pérols.</p>\n<p>venez au moment où vous le souhaitez, entre 9h00 et 10h30 pour bénéficier d'un échange individuel avec des conseillers de Bouscaren.</p>"

"timings : [{'begin': '2026-02-02T09:00:00+01:00', 'end': '2026-02-02T10:30:00+01:00'}]"

In [14]:
mask = df["first_date"] >= '2026-02-01T00:00:00+01:00'
df_2026 = df[mask].sort_values('first_date')
df_2026

,uid,title_fr,description_fr,longdescription_fr,timings,first_date,location_name,location_city,location_department,location_address,canonicalurl,conditions_fr,location_lat,location_lon
2500,36643268,Fête de l'Ours à Arles sur Tech,La Fête de l’Ours d’Arles sur Tech représente ...,<p>La Fête de l’Ours d’Arles sur Tech représen...,"[{'begin': '2026-02-01T10:00:00+01:00', 'end':...",2026-02-01T10:00:00+01:00,"Rue du 14 juillet, Arles-sur-Tech",Arles-sur-Tech,Pyrénées-Orientales,"Rue du 14 juillet, 66150 Arles-sur-Tech",https://openagenda.com/pci-occitanie/events/fe...,Fête participative dont il est vivement conse...,42.456850,2.637150
7357,90245560,"""Prêt pour la saison ? Testez vos talents de r...",Venez tester vos habiletés sur le poste de réc...,<p>Venez tester vos habiletés sur le poste de ...,"[{'begin': '2026-02-02T08:45:00+01:00', 'end':...",2026-02-02T08:45:00+01:00,Agence VAUVERT,Vauvert,Gard,30600 Vauvert,https://openagenda.com/semaine-des-metiers-du-...,,43.707302,4.268453
702,17537675,"Serveur débutant, participez à notre Méthode d...",Vous débutez dans le monde de la restauration ...,<p>Vous débutez dans le monde de la restaurati...,"[{'begin': '2026-02-02T09:00:00+01:00', 'end':...",2026-02-02T09:00:00+01:00,Les Cabannes - Agence CARMAUX,Les Cabannes,Tarn,81170 Les Cabannes,https://openagenda.com/semaine-des-metiers-du-...,,44.066486,1.939754
7869,95892978,"Rencontre Transport & Logistique, BTP, Sécurit...",Vous êtes en recherche d'emploi ou en projet d...,<p>Vous êtes en recherche d'emploi ou en proje...,"[{'begin': '2026-02-02T09:00:00+01:00', 'end':...",2026-02-02T09:00:00+01:00,Pérols - Agence MONTPELLIER MEDITERRANEE,Pérols,Hérault,34470 Pérols,https://openagenda.com/semaine-des-metiers-du-...,,43.582291,3.938849
1665,27653361,JOB CHEF quand votre passion devient un métier.,"Si la cuisine est votre passion, elle pourrait...","<p>Si la cuisine est votre passion, elle pourr...","[{'begin': '2026-02-02T09:00:00+01:00', 'end':...",2026-02-02T09:00:00+01:00,Agence NARBONNE,Narbonne,Aude,11100 Narbonne,https://openagenda.com/semaine-des-metiers-du-...,,43.154114,2.975137
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4415,58159093,SPORTS ET LOISIRS EQUIFUN - Domaine de CASTEL ...,R,"<p>Dans une nature verdoyante, au coeur du pay...","[{'begin': '2032-01-01T02:00:00+01:00', 'end':...",2032-01-01T02:00:00+01:00,Domaine de Castel Fizel,Caudiès-de-Fenouillèdes,Pyrénées-Orientales,66220 CAUDIES DE FENOUILLEDES,https://openagenda.com/catalogue-structures-ac...,,42.803386,2.399724
4243,55974849,Chalet Ma Neou,R+O,<p>Accueil de groupes scolaires été comme hive...,"[{'begin': '2032-01-01T02:00:00+01:00', 'end':...",2032-01-01T02:00:00+01:00,Chalet Ma Neou,Formiguères,Pyrénées-Orientales,Route de Mont Louis 66210 LES ANGLES,https://openagenda.com/catalogue-structures-ac...,,42.613256,2.102987
4092,54346567,Gîte des Grands Causses,O,<p>Le gîte se situe dans un quartier résidenti...,"[{'begin': '2032-01-01T02:00:00+01:00', 'end':...",2032-01-01T02:00:00+01:00,Gîte des Grands Causses,Millau,Aveyron,425 rue jean jacques rousseau 12100 millau,https://openagenda.com/catalogue-departemental...,,44.102200,3.058545
3657,49562140,BERLATS ACCUEIL DECOUVERTE,R,<p>Le centre d'accueil se situe en Hautes Terr...,"[{'begin': '2032-01-01T02:00:00+01:00', 'end':...",2032-01-01T02:00:00+01:00,BERLATS ACCUEIL DECOUVERTE,Berlats,Tarn,BERLATS,https://openagenda.com/catalogue-structures-ac...,,43.696504,2.564540


In [7]:
df.columns

Index(['uid', 'title_fr', 'description_fr', 'longdescription_fr', 'timings',
       'first_date', 'location_name', 'location_city', 'location_department',
       'location_address', 'canonicalurl', 'conditions_fr', 'location_lat',
       'location_lon'],
      dtype='str')

In [16]:
today = datetime.today()
today.strftime("%A %-d %B %Y")

'mardi 3 février 2026'

6

In [12]:
ligne = "Hérault"
accents = { 'a': ['à', 'ã', 'á', 'â'],
                    'e': ['é', 'è', 'ê', 'ë'],
                    'i': ['î', 'ï'],
                    'u': ['ù', 'ü', 'û'],
                    'o': ['ô', 'ö'] }
for (char, accented_chars) in accents.items():
    for accented_char in accented_chars:
        ligne = ligne.replace(accented_char, char)

In [13]:
ligne

'Herault'

In [ ]:
chunks = pd.read_json("../data/processed/processed_events_2026-02-03.json")
mask = chunks.duplicated('location_city')
chunks = chunks[~mask]['location_city'].sort_values()
chunks.to_list()

ValueError: keep must be either "first", "last" or False